<a href="https://colab.research.google.com/github/a-forty-two/EY-MarApr26/blob/main/10_vectorstores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<br>

# Retrieval-Augmented Generation with Vector Stores

<br>

In the previous notebook, we learned about embedding models and exercised some of their capabilities. We discussed their intended use cases of longer-form document comparison and found ways to use it as a backbone for more custom semantic comparisons. This notebook will progress these ideas toward the retrieval model's intended use case and explore how to build chatbot systems that rely on *vector stores* to automatically save and retrieve information.

<br>

### **Learning Objectives:**

- Understand how semantic-similarity-backed systems can facilitate easy-to-use retrieval formulations.

- Learn how to incorporate retrieval modules into your chat model systems for a retrieval-augmented generation (RAG) pipeline, which can be applied to tasks like document retrieval and conversation memory buffers.

<br>

### **Questions To Think About:**

- This notebook does not attempt to incorporate hierarchical reasoning or non-naive RAG (such as planning agents). Consider what modifications would be necessary to make these components work in an LCEL chain.

- Consider when it would be best to move your vector store solution into a scalable service and when a GPU will become necessary for optimization.

<br>

### **Environment Setup:**

In [20]:
#%%capture
## ^^ Comment out if you want to see the pip install process


%pip install -q langchain langchain-nvidia-ai-endpoints gradio rich
%pip install -q arxiv pymupdf faiss-cpu
%pip install -U langchain-community faiss-cpu

## If you encounter a typing-extensions issue, restart your runtime and try again
from langchain_nvidia_ai_endpoints import ChatNVIDIA
ChatNVIDIA.get_available_models()

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

In [21]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
import os
os.environ["NVIDIA_API_KEY"] = ""
# NVIDIAEmbeddings.get_available_models()
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")

# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1")

----

<br>

## Part 1: Summary of RAG Workflows

This notebook will explore several paradigms and derive reference code to help you approach some of the most common retrieval-augmented workflows. Specifically, the following sections will be covered (with the differences highlighted):

<br>

> ***Vector Store Workflow for Conversational Exchanges:***
- Generate semantic embedding for each new conversation.
- Add the message body to a vector store for retrieval.
- Query the vector store for relevant messages to fill in the LLM context.

<br>

> ***Modified Workflow for an Arbitrary Document:***
- **Divide the document into chunks and process them into useful messages.**
- Generate semantic embedding for each **new document chunk**.
- Add the **chunk bodies** to a vector store for retrieval.
- Query the vector store for relevant **chunks** to fill in the LLM context.
    - ***Optional:* Modify/synthesize results for better LLM results.**

<br>

> **Extended Workflow for a Directory of Arbitrary Documents:**
- Divide **each document** into chunks and process them into useful messages.
- Generate semantic embedding for each new document chunk.
- Add the chunk bodies to **a scalable vector database for fast retrieval**.
    - ***Optional*: Exploit hierarchical or metadata structures for larger systems.**
- Query the **vector database** for relevant chunks to fill in the LLM context.
    - *Optional:* Modify/synthesize results for better LLM results.

<br>

Some of the most important terminology surrounding RAG is covered in detail on the [**LlamaIndex Concepts page**](https://docs.llamaindex.ai/en/stable/getting_started/concepts.html), which itself is a great starting point for progressing towards the LlamaIndex loading and retrieving strategy. We highly recommend using it as a reference as you continue with this notebook and advise you to try out LlamaIndex after the course to consider the pros and cons firsthand!


<!-- > <img src="https://drive.google.com/uc?export=view&id=1cFbKbVvLLnFPs3yWCKIuzXkhBWh6nLQY" width=1200px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/data_connection_langchain.jpeg" width=1200px/>
>
> From [**Retrieval | LangChain**🦜️🔗](https://python.langchain.com/v0.1/docs/modules/data_connection/)

----

<br>

## **Part 2:** RAG for Conversation History

In our previous explorations, we delved into the capabilities of document embedding models and used them to embed, store, and compare semantic vector representations of text. Though we could motivate how to efficiently extend this into vector store land manually, the true beauty of working with a standard API is its strong incorporation with other frameworks that can already do the heavy lifting for us!

<br>

### **Step 1**: Getting A Conversation

Consider a conversation crafted using Llama-13B between a chat agent and a blue bear named Beras. This dialogue, dense with details and potential diversions, provides a rich dataset for our study:


In [3]:
conversation = [  ## This conversation was generated partially by an AI system, and modified to exhibit desirable properties
    "[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the rocky mountains?",
    "[Agent] The Rocky Mountains are a beautiful and majestic range of mountains that stretch across North America",
    "[Beras] Wow, that sounds amazing! Ive never been to the Rocky Mountains before, but Ive heard many great things about them.",
    "[Agent] I hope you get to visit them someday, Beras! It would be a great adventure for you!"
    "[Beras] Thank you for the suggestion! Ill definitely keep it in mind for the future.",
    "[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research online or watching documentaries about them."
    "[Beras] I live in the arctic, so I'm not used to the warm climate there. I was just curious, ya know!",
    "[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains and their significance!"
]

Using the manual embedding strategy from the previous notebook is still very viable, but we can also rest easy and let a **vector store** do all that work for us!

<br>

### **Step 2:** Constructing Our Vector Store Retriever

To streamline similarity queries on our conversation, we can employ a vector store to help keep track of passages for us! **Vector Stores**, or vector storage systems, abstract away most of the low-level details of the embedding/comparison strategies and provide a simple interface to load and compare vectors.


<!-- > <img src="https://drive.google.com/uc?export=view&id=1ZjwYbSZzsXK6ZP8O1-cY3BeRffV4oqzb" width=1000px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/vector_stores.jpeg" width=1200px/>
>
> From [**Vector Stores | LangChain**🦜️🔗](https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/)

<br>

In addition to simplifying the process from an API perspective, vector stores also implement connectors, integrations, and optimizations under the hood. In our case, we will start with the [**FAISS vector store**](https://python.langchain.com/docs/integrations/vectorstores/faiss), which integrates a LangChain-compatable Embedding model with the [**FAISS (Facebook AI Similarity Search)**](https://github.com/facebookresearch/faiss) library to make the process fast and scalable on our local machine!

**Specifically:**

1. We can feed our conversation into [**a FAISS vector store**](https://python.langchain.com/docs/integrations/vectorstores/faiss) via the `from_texts` constructor. This will take our conversational data and the embedding model to create a searchable index over our discussion.
2. This vector store can then be "interpreted" as a retriever, supporting the LangChain runnable API and returning documents retrieved via an input query.

The following shows how you can construct a FAISS vector store and reinterpret it as a retriever using the LangChain `vectorstore` API:

In [22]:

%%time
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
# Updated import path
from langchain_community.vectorstores import FAISS

## Streamlined from_texts FAISS vectorstore construction from text list
convstore = FAISS.from_texts(conversation, embedding=embedder)
retriever = convstore.as_retriever()

CPU times: user 70.3 ms, sys: 238 µs, total: 70.6 ms
Wall time: 3.85 s


The retriever can now be used like any other LangChain runnable to query the vector store for some relevant documents:

In [23]:
pprint(retriever.invoke("What is your name?"))

[
    Document(
        id='f0bb9957-caa6-432b-abc9-d52a2f234a7e',
        metadata={},
        page_content="[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the 
rocky mountains?"
    ),
    Document(
        id='a711b4a3-3004-4cc3-85b6-ab6b07de2b16',
        metadata={},
        page_content='[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains
and their significance!'
    ),
    Document(
        id='ebf94c18-3448-4740-ab17-84809413ee0f',
        metadata={},
        page_content='[Agent] I hope you get to visit them someday, Beras! It would be a great adventure for 
you![Beras] Thank you for the suggestion! Ill definitely keep it in mind for the future.'
    ),
    Document(
        id='6c4f3826-8595-4d17-b264-4d03048712e2',
        metadata={},
        page_content="[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research 
online or watching documentaries about them.[Beras] I live in the arctic, so I'm not used to the warm climate 
there. I was just curious, ya know!"
    )
]

In [6]:
pprint(retriever.invoke("Where are the Rocky Mountains?"))

[
    Document(
        id='167bd1ba-43db-41a4-9fca-ceb99badb7ca',
        metadata={},
        page_content='[Agent] The Rocky Mountains are a beautiful and majestic range of mountains that stretch 
across North America'
    ),
    Document(
        id='e768f52e-3d16-42f8-9565-9a4458072eda',
        metadata={},
        page_content="[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research 
online or watching documentaries about them.[Beras] I live in the arctic, so I'm not used to the warm climate 
there. I was just curious, ya know!"
    ),
    Document(
        id='c2ba803d-2a78-4f24-a5ca-34b48c1df3a3',
        metadata={},
        page_content="[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the 
rocky mountains?"
    ),
    Document(
        id='8e097e3d-597a-4b8e-9a32-b5aa23706f05',
        metadata={},
        page_content='[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains
and their significance!'
    )
]

As we can see, our retriever found a handful of semantically relevant documents from our query. You may notice that not all of the documents are useful or clear on their own. For example, a retrieval of *"Beras"* for *"your name"* may be problematic for the chatbot if provided out of context. Anticipating the potential problems and creating synergies between your LLM components can increase the likelihood of good RAG behavior, so keep an eye out for such pitfalls and opportunities.

<br>

### **Step 3:** Incorporating Conversation Retrieval Into Our Chain

Now that we have our loaded retriever component as a chain, we can incorporate it into our existing chat system as before. Specifically, we can start with an ***always-on RAG formulation*** where:
- **A retriever is always retrieving context by default**.
- **A generator is acting on the retrieved context**.

In [24]:
from langchain_community.document_transformers import LongContextReorder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

from functools import partial
from operator import itemgetter

########################################################################
## Utility Runnables/Methods
def RPrint(preface=""):
    """Simple passthrough "prints, then returns" chain"""
    def print_and_return(x, preface):
        if preface: print(preface, end="")
        pprint(x)
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

def docs2str(docs, title="Document"):
    """Useful utility for making chunks into context string. Optional, but useful"""
    out_str = ""
    for doc in docs:
        doc_name = getattr(doc, 'metadata', {}).get('Title', title)
        if doc_name:
            out_str += f"[Quote from {doc_name}] "
        out_str += getattr(doc, 'page_content', str(doc)) + "\n"
    return out_str

## Optional; Reorders longer documents to center of output text
long_reorder = RunnableLambda(LongContextReorder().transform_documents)

In [38]:
context_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context."
    "\n\nRetrieved Context: {context}"
    "\n\nUser Question: {question}"
    "\nAnswer the user conversationally. User is not aware of context."
)

chain = (
    {
        'context': convstore.as_retriever() | long_reorder | docs2str,
        'question': (lambda x:x) # RunnablePassthrough is the same thing!
    }
    | context_prompt
    | RPrint()
    | instruct_llm
    | StrOutputParser()
)

pprint(chain.invoke("Where does Beras live?"))

ChatPromptValue(
    messages=[
        HumanMessage(
            content="Answer the question using only the context.\n\nRetrieved Context: [Quote from Document] [User]
Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the rocky mountains?\n[Quote from 
Document] [Agent] I hope you get to visit them someday, Beras! It would be a great adventure for you![Beras] Thank 
you for the suggestion! Ill definitely keep it in mind for the future.\n[Quote from Document] [Beras] Wow, that 
sounds amazing! Ive never been to the Rocky Mountains before, but Ive heard many great things about them.\n[Quote 
from Document] [Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research online 
or watching documentaries about them.[Beras] I live in the arctic, so I'm not used to the warm climate there. I was
just curious, ya know!\n\n\nUser Question: Where does Beras live?\nAnswer the user conversationally. User is not 
aware of context.",
            additional_kwargs={},
            response_metadata={}
        )
    ]
)

 According to the conversation, it appears that Beras lives in the Arctic.

Take a second to try out some more invocations and see how the new setup performs. Regardless of your model choice, the following questions should serve as interesting starting points.

In [30]:
pprint(chain.invoke("Where are the minions?")) # Info doesnt exist in our context (vectors)

ChatPromptValue(
    messages=[
        HumanMessage(
            content="Answer the question using only the context.\n\nRetrieved Context: [Quote from Document] [User]
Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the rocky mountains?\n[Quote from 
Document] [Agent] I hope you get to visit them someday, Beras! It would be a great adventure for you![Beras] Thank 
you for the suggestion! Ill definitely keep it in mind for the future.\n[Quote from Document] [Agent] The Rocky 
Mountains are a beautiful and majestic range of mountains that stretch across North America\n[Quote from Document] 
[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research online or watching 
documentaries about them.[Beras] I live in the arctic, so I'm not used to the warm climate there. I was just 
curious, ya know!\n\n\nUser Question: Where are the minions?\nAnswer the user conversationally. User is not aware 
of context.",
            additional_kwargs={},
            response_metadata={}
        )
    ]
)

 I'm sorry, but there doesn't seem to be any mention of minions in our conversation or the provided context. Could 
you please clarify or give more details so I can assist you better?

In [10]:
pprint(chain.invoke("Where are the Rocky Mountains? Are they close to California?"))

 Hello there! The Rocky Mountains are indeed a majestic range, stretching across a significant part of North 
America. While they are not directly in California, they are relatively close in a broader sense. The Rocky 
Mountains primarily run through Canada and the United States, covering parts of states like Montana, Idaho, 
Wyoming, Colorado, and New Mexico. They don't directly border California, but if you're traveling from California, 
you would be able to reach the Rockies after crossing through Nevada or Arizona into Utah and Colorado. Would you 
like to know more about any specific aspect of the Rocky Mountains?

In [11]:
pprint(chain.invoke("How far away is Beras from the Rocky Mountains?"))

 While the context does not provide information about Beras's exact location or distance from the Rocky Mountains, 
it does mention that Beras lives in the arctic. The Rocky Mountains are located in North America, stretching from 
western Canada and the United States. So, Beras would be quite far away from the Rocky Mountains since the arctic 
and North America are different regions.

<br>

You might notice some decent performance with this always-on retrieval node in the loop since the actual context being fed into the LLM remains relatively small. It's important to experiment with factors like embedding sizes, context limits, and model options to see what kinds of behavior you can expect and which efforts are worth taking to improve performance.

<br>

### **Step 4:** Automatic Conversation Storage

Now that we see how our vector store memory unit should function, we can perform one last integration to allow our conversation to add new entries to our conversation: a runnable that calls the `add_texts` method for us to update the store state.


In [39]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

########################################################################
## Reset knowledge base and define what it means to add more messages.
convstore = FAISS.from_texts(conversation, embedding=embedder)

def save_memory_and_get_output(d, vstore):
    """Accepts 'input'/'output' dictionary and saves to convstore"""
    vstore.add_texts([f"User said {d.get('input')}", f"Agent said {d.get('output')}"])
    return d.get('output')

########################################################################

# instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1")

chat_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context"
    "\n\nRetrieved Context: {context}"
    "\n\nUser Question: {input}"
    "\nAnswer the user conversationally. Make sure the conversation flows naturally.\n"
    "[Agent]"
)


conv_chain = (
    {
        'context': convstore.as_retriever() | long_reorder | docs2str,
        'input': (lambda x:x)
    }
    | RunnableAssign({'output' : chat_prompt | instruct_llm | StrOutputParser()})
    | partial(save_memory_and_get_output, vstore=convstore)
)

pprint(conv_chain.invoke("I'm glad you agree! I can't wait to get some ice cream there! It's such a good food!"))
print()
pprint(conv_chain.invoke("Can you guess what my favorite food is?"))
print()
pprint(conv_chain.invoke("Actually, my favorite is honey! Not sure where you got that idea?"))
print()
pprint(conv_chain.invoke("I see! Fair enough! Do you know my favorite food now?"))

 Oh, I wish we could enjoy some ice cream in the Rocky Mountains together! It would be quite an experience, 
wouldn't it? Although, let's focus on learning more about these beautiful mountains first, and then perhaps we can 
discuss our delicious plans!

 Based on our conversation, it seems you have a particular fondness for ice cream! I must say, it's a delightful 
treat, especially when imagined in the scenic backdrop of the Rocky Mountains. So, I'm guessing your favorite food 
is ice cream. Is that right?

 I see, I apologize for the confusion! It appears we took a different path from our conversation. Now that you've 
mentioned it, honey is indeed a wonderful food with a rich history and many health benefits. It's fascinating to 
think about how it's made by honeybees! Would you like to share what you love most about honey?

 Based on our recent conversation, it seems we had a little misunderstanding earlier. Now that we've clarified, I 
can confidently say that your favorite food is indeed honey! It's always a joy to learn about people's favorite 
foods and the stories behind them. So, do tell, what makes honey your top pick?

Unlike the more automatic full-text or rule-based approaches to injecting context into the LLM, this approach ensures some amount of consolidation which can keep the context length from getting out of hand. It's still not a full-proof strategy on its own, but it's a stark improvement for unstructured conversations (and doesn't even require a strong instruction-tuned model to perform slot-filling).

In [46]:
# 1. Determine how many items are in the store
total_docs = convstore.index.ntotal

# 2. Perform a search for "everything"
# We use k=total_docs to ensure no document is left behind
docs = convstore.similarity_search(" ", k=total_docs)

# 3. Display the results
for i, doc in enumerate(docs):
    print(f"Message {i+1}: {doc.page_content}")

Message 1: Agent said  I see, I apologize for the confusion! It appears we took a different path from our conversation. Now that you've mentioned it, honey is indeed a wonderful food with a rich history and many health benefits. It's fascinating to think about how it's made by honeybees! Would you like to share what you love most about honey?
Message 2: [Agent] The Rocky Mountains are a beautiful and majestic range of mountains that stretch across North America
Message 3: [Agent] I hope you get to visit them someday, Beras! It would be a great adventure for you![Beras] Thank you for the suggestion! Ill definitely keep it in mind for the future.
Message 4: Agent said  Oh, I wish we could enjoy some ice cream in the Rocky Mountains together! It would be quite an experience, wouldn't it? Although, let's focus on learning more about these beautiful mountains first, and then perhaps we can discuss our delicious plans!
Message 5: Agent said  Based on our conversation, it seems you have a par